In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Installér og importér R-læsning
!pip install pyreadr
import pyreadr

# Læs .rds fil fra tidymodels
model_data = pyreadr.read_r("model_data_clean.rds")[None]

print(model_data.info())

# Target
target_col = "churn"

# Numeriske features
quant_cols = [
    "account_active_days",
    "previous_subscriptions",
    "previous_campaigns",
    "previous_trials",
    "newsletters_before_order",
    "newsletters_after_order",
    "age"
]

# Kategoriske features
cat_cols = [
    "koen",
    "permission_given_order",
    "permission_given_today",
    "kundetid_gruppe",
    "cluster_label"
]

# Target variabel
y = model_data[target_col].astype(int).values

# One-hot encoding af kategoriske variable
X_cat_dummies = pd.get_dummies(model_data[cat_cols], drop_first=False)

# Saml features
X_df = pd.concat(
    [
        model_data[quant_cols].reset_index(drop=True),
        X_cat_dummies.reset_index(drop=True)
    ],
    axis=1
)

# Train/test split
X_train_df, X_test_df, y_train, y_test = train_test_split(
    X_df, y, test_size=0.20, random_state=8, stratify=y
)

# Skalering af numeriske features
scaler = StandardScaler()

X_train_arr = X_train_df.copy()
X_test_arr = X_test_df.copy()

X_train_arr[quant_cols] = scaler.fit_transform(X_train_df[quant_cols])
X_test_arr[quant_cols] = scaler.transform(X_test_df[quant_cols])

# Konverter til numpy arrays
X_train = X_train_arr.values.astype(np.float32)
X_test = X_test_arr.values.astype(np.float32)

# Tjek data
print(X_train.shape)
print("Churn rate:", y_train.mean())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.3/788.3 kB 13.8 MB/s eta 0:00:00
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1259 entries, 0 to 1258
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   account_active_days       1259 non-null   int32   
 1   koen                      1259 non-null   category
 2   permission_given_order    1259 non-null   category
 3   permission_given_today    1259 non-null   category
 4   previous_subscriptions    1259 non-null   int32   
 5   previous_campaigns        1259 non-null   int32   
 6   previous_trials           1259 non-null   int32   
 7   newsletters_before_order  1259 non-null   int32   
 8   newsletters_after_order   1259 non-null   int32   
 9   churn                     1259 non-null   category
 10  early_churn               1259 non-null   float64 
 11  age                       1259 non-null   float64 
 12  kundetid_gruppe        

In [ ]:
# model_01
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, backend as K, callbacks, regularizers

K.clear_session()
tf.random.set_seed(8)

# Build model 01
# Input dimension kommer automatisk fra X_train
model_01 = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.3),
    layers.Dense(32, activation="relu", kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.2),
    layers.Dense(1, activation="sigmoid")
], name="baseline_nn")

model_01.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)

# Training
early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

print("Training Model 01 (Baseline)...")
history_01 = model_01.fit(
    X_train, y_train,
    epochs=150,
    batch_size=32,
    validation_split=0.20,
    callbacks=[early_stop],
    verbose=1
)

# Evaluation
metrics_01 = model_01.evaluate(X_test, y_test, verbose=0)

print(f"Model 01 TEST AUC: {metrics_01[2]:.4f}")
print(f"Model 01 TEST ACC: {metrics_01[1]:.4f}")

In [ ]:
# model_02
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, backend as K, callbacks, regularizers

K.clear_session()
tf.random.set_seed(8)

# Byg model_02
model_02 = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),

    # Første lag: bredt lag + batch normalization
    layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(0.002)),
    layers.BatchNormalization(),
    layers.Dropout(0.4),

    # Andet lag
    layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(0.002)),
    layers.Dropout(0.3),

    # Tredje lag
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.2),

    # Output lag
    layers.Dense(1, activation="sigmoid")
], name="complex_nn_model")

# Kompilér model med optimizer og ekstra metrics
model_02.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

# Callbacks til bedre træningskontrol
early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=5,
    min_lr=0.00001
)

# Træn model
print("Træner Model 02 (Komplekst neuralt netværk)...")
history_02 = model_02.fit(
    X_train, y_train,
    epochs=200,
    batch_size=32,
    validation_split=0.20,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

# Evaluer model
metrics_02 = model_02.evaluate(X_test, y_test, verbose=0)

print(f"Model 02 TEST AUC:    {metrics_02[2]:.4f}")
print(f"Model 02 TEST ACC:    {metrics_02[1]:.4f}")

Training Model 02 (Complex NN)...
Epoch 1/200
26/26 ━━━━━━━━━━━━━━━━━━━━ 6s 53ms/step - accuracy: 0.5391 - auc: 0.4796 - loss: 1.0565 - recall: 0.3050 - val_accuracy: 0.5495 - val_auc: 0.5256 - val_loss: 0.9290 - val_recall: 0.2716 - learning_rate: 0.0010
Epoch 2/200
26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step - accuracy: 0.6000 - auc: 0.6031 - loss: 0.9396 - recall: 0.4633 - val_accuracy: 0.6089 - val_auc: 0.5907 - val_loss: 0.9174 - val_recall: 0.2469 - learning_rate: 0.0010
Epoch 3/200
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.5963 - auc: 0.6170 - loss: 0.9189 - recall: 0.3754 - val_accuracy: 0.6337 - val_auc: 0.6425 - val_loss: 0.9086 - val_recall: 0.3951 - learning_rate: 0.0010
Epoch 4/200
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.6348 - auc: 0.6444 - loss: 0.8977 - recall: 0.4809 - val_accuracy: 0.6535 - val_auc: 0.6570 - val_loss: 0.8980 - val_recall: 0.4815 - learning_rate: 0.0010
Epoch 5/200
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.6435 - auc: 

In [ ]:
# model_03
from imblearn.over_sampling import SMOTE
import tensorflow as tf
from tensorflow.keras import layers, callbacks, regularizers, backend as K

K.clear_session()
tf.random.set_seed(8)

# Anvend SMOTE til at balancere træningsdata
sm = SMOTE(random_state=8)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

print(f"Oprindelig form: {X_train.shape}")
print(f"Ny form efter SMOTE: {X_train_res.shape}")

# Byg model 03 (trænet på balancerede data)
model_03 = tf.keras.Sequential([
    layers.Input(shape=(X_train_res.shape[1],)),

    # Første lag
    layers.Dense(32, activation="relu", kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.2),

    # Andet lag
    layers.Dense(16, activation="relu"),
    layers.Dropout(0.1),

    # Output lag
    layers.Dense(1, activation="sigmoid")
], name="nn_smote_balanced")

# Kompilér model
model_03.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)

# Træningsopsætning
early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=12,
    restore_best_weights=True
)

print("Træner Model 03 (SMOTE balanceret)...")
history_03 = model_03.fit(
    X_train_res, y_train_res,
    epochs=150,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=[early_stop],
    verbose=1
)

# Evaluér model
results_03 = model_03.evaluate(X_test, y_test, verbose=0)

print(f"MODEL 03 TEST AUC:        {results_03[2]:.4f}")
print(f"MODEL 03 TEST ACCURACY:   {results_03[1]:.4f}")

In [ ]:
# model_04
from imblearn.over_sampling import SMOTENC
import tensorflow as tf
from tensorflow.keras import layers, callbacks, regularizers, backend as K

K.clear_session()
tf.random.set_seed(8)

# Find indeks for kategoriske features (one-hot encoded)
categorical_cols = X_cat_dummies.columns.tolist()
cat_idx = [X_train_df.columns.get_loc(col) for col in categorical_cols]

# Anvend SMOTENC til blandede data (numerisk + kategorisk)
sm = SMOTENC(
    categorical_features=cat_idx,
    random_state=8,
    k_neighbors=5
)

X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

print(f"Oprindelig form: {X_train.shape}")
print(f"Form efter SMOTENC: {X_train_res.shape}")

# Byg model 04 (trænet på SMOTENC-balanceret data)
model_04 = tf.keras.Sequential([
    layers.Input(shape=(X_train_res.shape[1],)),

    # Første lag
    layers.Dense(32, activation="relu", kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.2),

    # Andet lag
    layers.Dense(16, activation="relu"),
    layers.Dropout(0.1),

    # Output lag
    layers.Dense(1, activation="sigmoid")
], name="nn_smotenc_balanced")

# Kompilér model
model_04.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)

# Træningsopsætning
early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=12,
    restore_best_weights=True
)

print("Træner Model 04 (SMOTENC balanceret)...")
history_04 = model_04.fit(
    X_train_res, y_train_res,
    epochs=150,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=[early_stop],
    verbose=1
)

# Evaluering
results_04 = model_04.evaluate(X_test, y_test, verbose=0)

print(f"MODEL 04 TEST AUC:      {results_04[2]:.4f}")
print(f"MODEL 04 TEST ACCURACY: {results_04[1]:.4f}")

In [ ]:
# Visualisering: sammenligning af modeller
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import confusion_matrix, roc_curve, auc, recall_score, accuracy_score

# Lav forudsigelser på testdata
probs_01 = model_01.predict(X_test)
probs_02 = model_02.predict(X_test)
probs_03 = model_03.predict(X_test)
probs_04 = model_04.predict(X_test)

# Konverter sandsynligheder til klasser (threshold = 0.5)
preds_03 = (probs_03 > 0.5).astype(int)
preds_04 = (probs_04 > 0.5).astype(int)

# ROC-kurver for alle modeller
plt.figure(figsize=(10, 7))

model_data = [
    (probs_01, 'Model 01: Baseline', 'blue'),
    (probs_02, 'Model 02: Kompleks', 'green'),
    (probs_03, 'Model 03: SMOTE', 'red'),
    (probs_04, 'Model 04: SMOTENC', 'purple')
]

for probs, label, color in model_data:
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, lw=2, label=f'{label} (AUC = {roc_auc:.2f})')

# Baseline diagonal
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC-kurve: model sammenligning')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

# Confusion matrices for SMOTE-modeller
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm_03 = confusion_matrix(y_test, preds_03)
sns.heatmap(cm_03, annot=True, fmt='d', cmap='Reds', cbar=False, ax=axes[0])
axes[0].set_title('Model 03 (SMOTE)')
axes[0].set_xlabel('Forudsagt')
axes[0].set_ylabel('Faktisk')

cm_04 = confusion_matrix(y_test, preds_04)
sns.heatmap(cm_04, annot=True, fmt='d', cmap='Purples', cbar=False, ax=axes[1])
axes[1].set_title('Model 04 (SMOTENC)')
axes[1].set_xlabel('Forudsagt')
axes[1].set_ylabel('Faktisk')

plt.tight_layout()
plt.show()

# Samlet performance tabel
results = []

for m_name, m_probs in [
    ("Model 01 (Baseline)", probs_01),
    ("Model 02 (Kompleks)", probs_02),
    ("Model 03 (SMOTE)", probs_03),
    ("Model 04 (SMOTENC)", probs_04)
]:
    m_preds = (m_probs > 0.5).astype(int)
    fpr, tpr, _ = roc_curve(y_test, m_probs)

    results.append({
        "Model": m_name,
        "AUC": round(auc(fpr, tpr), 3),
        "Recall (fangede churn)": round(recall_score(y_test, m_preds), 3),
        "Accuracy": round(accuracy_score(y_test, m_preds), 3)
    })

print("\nModel performance sammenligning")
print(pd.DataFrame(results).sort_values(by="AUC", ascending=False).to_string(index=False))

In [ ]:
import numpy as np
import pandas as pd

# Best case scenario

# 1. Her defineres inputdata for et "best case" scenarie
test_data = {
    "Aktive dage": 1000,
    "Tidligere subscriptions": 1,
    "tidligere kampagner": 1,
    "Tidligere trials": 0,
    "nyhedsbreve_før_ordre": 1,
    "nyhedsbreve_efter_ordre": 1,
    "Alder": 60,
    "Gender": "Mand",
    "Klynge": "Veteranen (Høj historik)",
    "Kundetid": "6+ måneder"
}

# 2. De numeriske variabler skaleres, så de matcher modelens træning
# (forudsætter at scaler allerede er trænet og tilgængelig i sessionen)
numeric_input = np.array([[
    test_data["Aktive dage"],
    test_data["Tidligere subscriptions"],
    test_data["tidligere kampagner"],
    test_data["Tidligere trials"],
    test_data["nyhedsbreve_før_ordre"],
    test_data["nyhedsbreve_efter_ordre"],
    test_data["Alder"]
]])
scaled_numeric = scaler.transform(numeric_input)

# 3. Der oprettes en tom feature-matrix med samme struktur som modellen forventer
# Her er alle værdier sat til 0 som udgangspunkt
X_verify = np.zeros((1, 23))
X_verify[0, 0:7] = scaled_numeric

# 4. De kategoriske variabler indsættes manuelt, så de matcher encoding-strukturen
# (skal stemme overens med den mapping der er brugt i Power BI / træning)

# Køn
if test_data["Gender"] == "Mand":
    X_verify[0, 7] = 1
elif test_data["Gender"] == "Kvinde":
    X_verify[0, 8] = 1

# Klynge
if test_data["Klynge"] == "Veteranen (Høj historik)":
    X_verify[0, 14] = 1

# Kundetid
if test_data["Kundetid"] == "6+ måneder":
    X_verify[0, 19] = 1

# 5. Til sidst laves en forudsigelse med den trænede model
colab_prediction = model_04.predict(X_verify)
print(f"NN Prediction af best case: {colab_prediction[0][0] * 100:.2f}%")

In [ ]:
import numpy as np
import pandas as pd

# Worst case scenario

# 1. Her opstilles et "worst case" scenarie, hvor kunden har lav aktivitet og lav historik
test_data_high_risk = {
    "Aktive dage": 10,
    "Tidligere subscriptions": 0,
    "tidligere kampagner": 0,
    "Tidligere trials": 0,
    "nyhedsbreve_før_ordre": 0,
    "nyhedsbreve_efter_ordre": 0,
    "Alder": 25,
    "Gender": "Mand",
    "Klynge": "Mobil-nykommeren",
    "Kundetid": "0-7 dage"
}

# 2. De numeriske værdier skaleres, så de passer til modellens inputformat
numeric_input = np.array([[
    test_data_high_risk["Aktive dage"],
    test_data_high_risk["Tidligere subscriptions"],
    test_data_high_risk["tidligere kampagner"],
    test_data_high_risk["Tidligere trials"],
    test_data_high_risk["nyhedsbreve_før_ordre"],
    test_data_high_risk["nyhedsbreve_efter_ordre"],
    test_data_high_risk["Alder"]
]])
scaled_numeric = scaler.transform(numeric_input)

# 3. Der oprettes en tom feature-vektor med samme struktur som modellen forventer
# Alle værdier initialiseres til 0, hvorefter relevante features sættes manuelt
X_verify = np.zeros((1, 23))
X_verify[0, 0:7] = scaled_numeric

# 4. De kategoriske variable indkodes manuelt, så de matcher encoding-strukturen fra Power BI

# Køn
if test_data_high_risk["Gender"] == "Mand":
    X_verify[0, 7] = 1
elif test_data_high_risk["Gender"] == "Kvinde":
    X_verify[0, 8] = 1

# Klynge: Mobil-nykommeren er placeret på index 10 i den anvendte mapping
X_verify[0, 10] = 1

# Kundetid: 0-7 dage er placeret på index 16 i mappingen
X_verify[0, 16] = 1

# 5. Til sidst foretages en forudsigelse med den trænede model
colab_pred = model_04.predict(X_verify, verbose=0)
print(f"NN Prediction af worst case: {colab_pred[0][0] * 100:.2f}%")